# TaaS Model & Agentic Studio — CPU / CUDA / Ollama Quick Launch


**Hypergraph integration:** Hypergraph Explorer is part of the Studio build. Its **Run discovery** action reuses the authenticated Core discovery workflow and displays the server-generated projection. This notebook uses one checkout, one Studio process, one token and one Colab proxy; it does not install or use browser automation.
**Operator guide:** [Integrated Colab quickstart](../../docs/deployment/COLAB_T4_QUICKSTART.md)

This notebook launches the integrated Model/Agent/TaaS Studio from an immutable repository revision inside the current Colab runtime.

**Available in this pinned revision:** Hugging Face model search and metadata inspection, explicit model loading, real model-backed chat inference, model-assisted GoalSpec proposals, TaaS tool discovery and pre-execution assurance, and configured MCP stdio server/tool inspection.

**Important boundaries:** MCP inspection does not invoke tools; TaaS certification is pre-execution evidence; the Agent workspace does not yet run an autonomous tool-calling loop. Persistent run history, streaming traces, multi-user isolation and production deployment are not provided.

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, secrets

from IPython.display import display, HTML



REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"

TAAS_COMMIT = "69a1c8e10709d5434166a553fab7ea3fb4e71336"

WORKDIR = pathlib.Path("/content/taas-provenance-discovery")

APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000



if not WORKDIR.exists():

    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)

subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)

actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()

assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)

assert (WORKDIR / "pyproject.toml").is_file()

assert (WORKDIR / "src" / "taas").is_dir()

print("TaaS revision:", actual)

print("TaaS API source:", WORKDIR / "src" / "taas" / "studio" / "api.py")

## Hardware detection

A T4 is useful for local model inference when Colab allocates one. The TaaS discovery engine itself is CPU-oriented and does not require a GPU.

The model runtime loads weights in the same Python process that hosts this Studio API. A successful GPU probe is not a model-load test; verify actual load and inference from Model Studio. If no GPU is available, CPU mode may still work for sufficiently small models.

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
GPU_AVAILABLE = gpu_check.returncode == 0
print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; Hugging Face CPU and Ollama remain available.")
print("Python:", sys.version.split()[0])

## Install and verify TaaS Studio

Install the Studio API and optional Hugging Face inference dependencies into the active Colab interpreter. The model runtime requires the `hf` extra (Transformers, PyTorch, Accelerate and Safetensors). If installation or import verification fails, the notebook stops with the diagnostic output.

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[test,studio,hf]"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(install.stdout[-4000:])
if install.returncode:
    print(install.stderr[-4000:])
    raise RuntimeError("TaaS Studio installation failed")
import_check = subprocess.run(
    [
        sys.executable, "-c",
        "import traceback, importlib; "
        "importlib.import_module('taas'); "
        "importlib.import_module('fastapi'); "
        "importlib.import_module('uvicorn'); "
        "importlib.import_module('torch'); "
        "importlib.import_module('transformers'); "
        "importlib.import_module('safetensors'); "
        "importlib.import_module('sentencepiece'); "
        "importlib.import_module('tiktoken'); "
        "importlib.import_module('taas.studio.ollama_runtime'); "
        "importlib.import_module('taas.studio.api'); "
        "print('TaaS Model/Agent Studio imports OK')"
    ],
    cwd=WORKDIR, text=True, capture_output=True
)
print(import_check.stdout)
if import_check.returncode:
    print("=== IMPORT CHECK STDERR / TRACEBACK ===")
    print(import_check.stderr[-12000:])
    print("Python executable:", sys.executable)
    print("Working directory:", WORKDIR)
    print("Package source exists:", (WORKDIR / "src" / "taas" / "__init__.py").is_file())
    raise RuntimeError("TaaS Model/Agent Studio import check failed; see traceback above")

## Optional: enable Ollama in this Colab runtime

By default, skip the next cell and use Hugging Face CPU/CUDA. To test Ollama, run the optional setup cell **before** launching Studio. It installs Ollama, starts its daemon in this VM, and pulls `qwen2.5:0.5b`. This is separate from the Hugging Face loader and can consume additional RAM/disk. See [Model Runtime Backends](../../docs/deployment/MODEL_RUNTIME_BACKENDS.md).

In [ ]:
ENABLE_OLLAMA = False  # Change to True only when you want the optional Ollama backend
if ENABLE_OLLAMA:
    if not shutil.which("ollama"):
        subprocess.run(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)
    ollama_env = os.environ.copy()
    ollama_env["OLLAMA_HOST"] = "127.0.0.1:11434"
    if "ollama_process" not in globals() or ollama_process.poll() is not None:
        ollama_log = open("/content/ollama.log", "a", buffering=1)
        ollama_process = subprocess.Popen(["ollama", "serve"], env=ollama_env, stdout=ollama_log, stderr=subprocess.STDOUT)
    for attempt in range(45):
        try:
            urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError("Ollama daemon did not become ready; inspect /content/ollama.log")
    subprocess.run(["ollama", "pull", "qwen2.5:0.5b"], check=True)
    os.environ["TAAS_OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"
    print("Ollama is ready. Select Ollama in the Studio runtime panel.")
else:
    print("Ollama setup skipped. Hugging Face CPU/CUDA remains available.")


## Launch Studio

An ephemeral API token is generated automatically for this Colab runtime. Copy it into the Studio token field; it is not committed or persisted by this notebook.

The default TaaS model profile remains **strict and uncalibrated** unless an operator explicitly supplies a measured `TAAS_MODEL_PROFILE`. Installing or loading a Hugging Face model does not manufacture capability calibration. Unknown native capabilities therefore remain unassumed.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
print("Temporary Studio API token (copy into Studio; do not share):")
print(api_token)

# Make rerunning this cell safe, including after the Stop cell was used.
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.close()

studio_log = open("/content/taas-studio.log", "a", buffering=1)
studio_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",
     "--host", APP_BIND_HOST, "--port", str(APP_PORT)],
    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT
)

def wait_http(url, attempts=40):
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                return response.status, response.read().decode("utf-8")
        except Exception as exc:
            last = exc
            time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health"))
print("Readiness:", wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/ready"))
print("Studio process PID:", studio_process.pid)


## Open TaaS Studio

Run this cell and open the link. Keep the Colab runtime connected while using Studio.

In [ ]:
from google.colab import output

# Never reuse a proxy URL cached by an earlier server/runtime lifecycle.
if "studio_process" not in globals() or studio_process.poll() is not None:
    raise RuntimeError("TaaS Studio is not running. Execute the Launch Studio cell and inspect /content/taas-studio.log.")
status, health_body = wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health")
if status != 200:
    raise RuntimeError(f"TaaS Studio health check failed: HTTP {status}: {health_body}")
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
if not proxy_url or not proxy_url.startswith("https://"):
    raise RuntimeError(f"Colab did not return a valid HTTPS proxy URL: {proxy_url!r}")
# Add a per-launch cache key so an old Colab proxy response cannot be reused.
proxy_separator = "&" if "?" in proxy_url else "?"
fresh_proxy_url = proxy_url + proxy_separator + "studio_session=" + str(time.time_ns())
display(HTML(f'<p><a href="{fresh_proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))
print("Studio health: HTTP", status)
print("Studio PID:", studio_process.pid)
print("Fresh Colab proxy URL generated (cache-busted) for this running server.")


## Optional · public Cloudflare tunnel for remote/agent-driven E2E

**Use only when an external operator or browser agent must reach this Studio.** The Colab `proxyPort` URL above works only inside *your* Google session — other browsers get HTTP 404. This cell publishes the Studio through a short-lived Cloudflare quick tunnel (`*.trycloudflare.com`) so a remote browser can drive it end-to-end.

- The quick-tunnel URL is **public but not an authentication mechanism** — every API route still requires the `TAAS_STUDIO_API_TOKEN` bearer token printed by the launch cell. Never share that token publicly.
- `/`, `/health`, `/ready` are unauthenticated at the origin — run this only for a short, supervised test session, then run **Stop Studio**.
- For anything beyond a throwaway test, use a **named tunnel + Cloudflare Access policy** as documented in `COLAB_T4_CORE_STUDIO.md` instead.
- cloudflared is downloaded from Cloudflare's release CDN each run; this is a convenience path, not a pinned supply chain.


In [ ]:
import os, re, subprocess, time

ENABLE_QUICK_TUNNEL = False  # set True only for a short supervised remote/E2E session

if not ENABLE_QUICK_TUNNEL:
    print("Quick tunnel disabled. Set ENABLE_QUICK_TUNNEL = True to publish the Studio for a remote E2E session.")
elif "studio_process" not in globals() or studio_process.poll() is not None:
    raise RuntimeError("TaaS Studio is not running. Execute the Launch Studio cell first.")
else:
    cf_bin = "/content/cloudflared"
    if not os.path.exists(cf_bin):
        dl = subprocess.run(
            ["bash", "-lc",
             "curl -fsSL -o /content/cloudflared "
             "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
             "&& chmod +x /content/cloudflared"],
            text=True, capture_output=True
        )
        if dl.returncode:
            print(dl.stdout[-2000:]); print(dl.stderr[-2000:])
            raise RuntimeError("cloudflared download failed")
    tunnel_log = open("/content/cloudflared.log", "a", buffering=1)
    if "tunnel_process" in globals() and tunnel_process.poll() is None:
        tunnel_process.terminate(); tunnel_process.wait(timeout=10)
    tunnel_process = subprocess.Popen(
        [cf_bin, "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}", "--no-autoupdate"],
        stdout=tunnel_log, stderr=subprocess.STDOUT
    )
    public_url = None
    for _ in range(30):
        time.sleep(1)
        if tunnel_process.poll() is not None:
            break
        try:
            with open("/content/cloudflared.log") as fh:
                match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", fh.read())
            if match:
                public_url = match.group(0); break
        except OSError:
            pass
    if not public_url:
        raise RuntimeError("cloudflared did not publish a URL; inspect /content/cloudflared.log")
    print("Public Studio URL (share ONLY with the tester, together with the API token):", public_url)
    print("Reminder: the quick tunnel is public but every API route still requires the bearer token.")


## Core-first Objective Engineering acceptance checks

This cell verifies the canonical objective endpoints, generic intent contract, exact Theta type enforcement, Core-first UI workflow, and the oncology objective regression (three outcomes, November 2024 interval, top-one publication ordering, and article dependencies). It does not invoke tools.

After loading a model in Studio, set `RUN_LIVE_OBJECTIVE_TEST = True` to send one real prompt through model decomposition, structural validation, objective conformance and TaaS Core `gap_analysis`. A rejected proposal is reported as a failed acceptance test; it is never auto-applied or executed.


In [ ]:
import json
import urllib.request

# Ensure the repository's src-layout package is importable even if editable
# installation was skipped or the Colab runtime was restarted.
from pathlib import Path
import sys
_taas_src = Path(WORKDIR) / "src"
if not (_taas_src / "taas" / "__init__.py").is_file():
    raise RuntimeError(
        f"TaaS source tree not found at {_taas_src}. Run the first setup cell "
        "from the beginning to clone and pin the repository."
    )
if str(_taas_src) not in sys.path:
    sys.path.insert(0, str(_taas_src))
print("TaaS import source:", _taas_src)

import importlib
importlib.invalidate_caches()
import taas.studio.objective_contract as objective_contract
importlib.reload(objective_contract)  # avoid stale module objects in reused Colab kernels
derive_objective_contract = objective_contract.derive_objective_contract
from taas.studio.objective_contract import build_outcome_coverage
from taas.studio.analyst_contract import validate_theta_types

# Colab kernels may retain an older installed/module-cached Studio API after
# switching repository revisions. Reload the module from the verified checkout.
import taas.studio.api as studio_api
studio_api = importlib.reload(studio_api)
required_agent_api = (
    "canonicalize_declared_plural_types",
    "canonicalize_objective_constraint_dependencies",
    "canonicalize_retrieval_target_inputs",
    "canonicalize_self_dependencies",
    "canonicalize_arithmetic_expressions_from_effects",
    "prepare_agent_candidate",
    "validate_studio_goal_spec",
)
missing_agent_api = [name for name in required_agent_api if not hasattr(studio_api, name)]
if missing_agent_api:
    api_path = Path(studio_api.__file__).resolve()
    raise ImportError(
        f"Studio API at {api_path} is missing {missing_agent_api}. "
        f"Expected checkout {actual}; restart runtime and run all cells from the beginning."
    )
app = studio_api.app
canonicalize_declared_plural_types = studio_api.canonicalize_declared_plural_types
canonicalize_objective_constraint_dependencies = studio_api.canonicalize_objective_constraint_dependencies
prepare_agent_candidate = studio_api.prepare_agent_candidate
validate_studio_goal_spec = studio_api.validate_studio_goal_spec
print("Studio API reload OK:", studio_api.__file__)

# A retrieval-only gap is classified as a tool requirement, not a model-skill gap.
from types import SimpleNamespace
retrieval_only = studio_api.classify_core_gap_requirements(
    [SimpleNamespace(id="g1", cls="retrieve")], ["g1"]
)
core = {"status": retrieval_only.get("status")}
assert core.get("status") == "tool_requirements_detected", retrieval_only
assert retrieval_only.get("tool_required_goal_ids") == ["g1"], retrieval_only
assert retrieval_only.get("model_capability_gap_goal_ids") == [], retrieval_only
print("Retrieval-only Core gap classification OK")

# 1) Canonical API and UI surface
paths = set(app.openapi()["paths"])
assert "/api/v1/objectives/prepare" in paths
assert "/api/v1/objectives/discover" in paths
assert "/api/v1/agents/decompose" not in paths

page = urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/").read().decode("utf-8")
assert 'runtimePost("/api/v1/objectives/prepare"' in page
assert 'fetch("/api/v1/objectives/discover"' in page
assert 'id="agent-route-prompt-core"' in page

# 2) Domain-independent objective contract smoke tests
cases = [
    ("Find the latest research papers about federated learning.", "search"),
    ("Summarise the document supplied in the prompt.", "summarise"),
    ("Translate this paragraph into French.", "translate"),
    ("Count all invoices issued this year.", "count"),
    ("Calculate average sales by region.", "aggregate"),
    ("Compare these two provided architectures.", "compare"),
    ("Identify anomalies in the supplied sensor data.", "classify"),
    ("Create a report from several external sources.", "create_document"),
    ("Generate an image illustrating the architecture.", "render_media"),
    ("Send an email notification to the project owner.", "send"),
]
contracts = {}
for prompt, expected_class in cases:
    contract = derive_objective_contract(prompt)
    contracts[prompt] = contract
    assert expected_class in contract["required_goal_classes"], (prompt, contract)
    print(expected_class, "OK:", prompt)

# Distinguish supplied content from external-source work. This prevents
# unnecessary retrieval/discovery for local transformations and comparisons.
for prompt in (
    "Summarise the document supplied in the prompt.",
    "Translate this paragraph into French.",
    "Compare these two provided architectures.",
    "Identify anomalies in the supplied sensor data.",
):
    contract = contracts[prompt]
    assert contract["contract_version"] == "2.0", (prompt, contract)
    assert contract["user_supplied_content"] is True, (prompt, contract)
    assert contract["source_data_required"] is False, (prompt, contract)

for prompt in (
    "Find the latest research papers about federated learning.",
    "Count all invoices issued this year.",
    "Calculate average sales by region.",
    "Create a report from several external sources.",
):
    contract = contracts[prompt]
    assert contract["source_data_required"] is True, (prompt, contract)

print("Supplied-content vs external-source routing checks OK")

# 2b) Objective Contract v2: versioned outcomes and scenario-independent frames.
# Each lexical outcome is a proposal for Core validation, not a semantic proof.
for prompt, expected in cases:
    frame = contracts[prompt]
    assert frame["contract_version"] == "2.0", (prompt, frame)
    assert [item["goal_class"] for item in frame["outcomes"]] == frame["required_goal_classes"]
    assert all(item["required"] for item in frame["outcomes"])
    assert frame["review"]["semantic_correctness_verified"] is False
    assert "inputs" in frame["constraints"] and "scope" in frame["constraints"]

compound = derive_objective_contract(
    "Search for recent papers on formal concept analysis, compare the findings, "
    "synthesise the evidence, and create a report."
)
assert [item["goal_class"] for item in compound["outcomes"]] == [
    "search", "compare", "synthesise", "create_document"
], compound
assert compound["constraints"]["inputs"]["source_data_required"] is True
assert compound["review"]["semantic_correctness_verified"] is False
print("Objective Contract v2 compound-outcome checks OK:", [
    item["goal_class"] for item in compound["outcomes"]
])

# 2c) Regression: year difference requires retrieval plus exact computation.
assaultcube_objective = (
    "Can you get me the year difference between the creation date of the "
    "AssaultCube repository and the domain registration date of its official site?"
)
assaultcube_contract = derive_objective_contract(assaultcube_objective)
assert assaultcube_contract["required_goal_classes"] == ["retrieve", "compute_exact"], assaultcube_contract
assert assaultcube_contract["numeric_result_required"] is True, assaultcube_contract
assert assaultcube_contract["exactness_required"] == "exact", assaultcube_contract
assaultcube_coverage = build_outcome_coverage(assaultcube_objective, {
    "goals": [
        {"id": "g1", "class": "retrieve"},
        {"id": "g2", "class": "retrieve"},
    ]
})
assert assaultcube_coverage["summary"] == {"required": 2, "covered": 1, "missing": 1}
assert assaultcube_coverage["outcomes"][1]["goal_class"] == "compute_exact"
assert assaultcube_coverage["outcomes"][1]["status"] == "missing"
print("Year-difference contract and retrieval-only rejection checks OK")

# 2d) Regression checks: source prepositions are not date ranges, while
# explicitly date-shaped English/French ranges remain temporal constraints.
source_objective = "Retrieve records matching the provided query from the configured source."
source_frame = derive_objective_contract(source_objective)
assert source_frame["operation"] == "retrieve", source_frame
assert source_frame["temporal_scope"] is None, source_frame
assert not ({"configured", "matching", "query", "source"} & set(source_frame["unresolved_business_concepts"])), source_frame

for range_prompt, expected_expression in (
    ("Retrieve records from 1 January to 31 January.", "from_to"),
    ("Retrieve records between January 1 and January 31.", "between"),
    ("Récupérer les dossiers du 1 janvier au 31 janvier.", "du_au"),
    ("Retrieve records between 2026-01-01 and 2026-01-31.", "between"),
):
    temporal_frame = derive_objective_contract(range_prompt)
    assert temporal_frame["temporal_scope"]["kind"] == "explicit_range", (range_prompt, temporal_frame)
    assert temporal_frame["temporal_scope"]["expression"] == expected_expression, (range_prompt, temporal_frame)
print("Objective temporal false-positive and explicit-range checks OK")

# Run the Objective Contract v2 and GoalSpec conformance suites against the exact pinned checkout.
v2_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_objective_contract_v2.py", "tests/test_goalspec_conformant_agent.py", "tests/test_studio_api.py", "tests/test_agent_candidate_normalization.py", "tests/test_studio_objective_constraint_dependencies.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(v2_tests.stdout)
if v2_tests.returncode:
    print(v2_tests.stderr[-8000:])
    raise RuntimeError("Objective Contract v2 pytest acceptance failed")


french = derive_objective_contract("Compter toutes les factures émises cette année.")
assert french["operation"] == "count"
assert french["temporal_scope"]["expression"] == "this_year"
assert french["exactness_required"] == "exact"

grouped = derive_objective_contract("Calculate average sales by region.")
assert grouped["grouped_result_required"] is True
assert grouped["numeric_result_required"] is False

# 3) Exact Theta vocabulary enforcement
invalid_type = {
    "prompt_inputs": ["text"],
    "goals": [{
        "id": "g1",
        "class": "retrieve",
        "in": ["query"],
        "out": "record:invoice[]",
        "desc": "Retrieve invoice records",
    }],
}
theta_error = validate_theta_types(invalid_type)
assert theta_error and "undeclared" in theta_error
print("Theta rejects undeclared type combinations OK")

# Effect vocabulary is not inferred from implementation verbs: ordinary retrieval
# reads a source; execute(query) is not a declared effect and must remain rejected.
from taas.studio.analyst_contract import build_analyst_system_prompt, build_refinement_context
analyst_prompt = build_analyst_system_prompt()
refinement_prompt = build_refinement_context(
    "Retrieve records matching the provided query from the configured source.",
    '{"eff":["execute(query)"]}',
    [{"code": "goalspec_structural_nonconformance", "severity": "error",
      "message": "invalid effect execute(query)"}],
)
for prompt_text in (analyst_prompt, refinement_prompt):
    assert "execute(query)" in prompt_text
    assert "read(source)" in prompt_text
    assert "backend implementation details" in prompt_text
print("Canonical retrieval effect guidance checks OK")

# 2d) Agent proposal boundary: repair only an explicitly declared Θ plural alias,
# preserve the exact user objective, and retain raw model text for audit.
normalization_input = {
    "prompt_inputs": ["text"],
    "goals": [{
        "id": "g1", "class": "retrieve", "desc": "Retrieve records",
        "in": ["query"], "out": "records[]", "deps": [],
    }],
}
normalized, normalization_events = canonicalize_declared_plural_types(normalization_input)
assert normalized["goals"][0]["out"] == "record[]"
assert normalization_events[0]["from"] == "records[]"
assert normalization_events[0]["to"] == "record[]"
assert normalization_input["goals"][0]["out"] == "records[]"

objective = "Retrieve records matching the provided query from the configured source."
prepared = prepare_agent_candidate({
    "text": '{"prompt":"Retrieve records matching the provided query from the configured sourc"}',
    "goal_spec": {
        "prompt": "Retrieve records matching the provided query from the configured sourc",
        **normalization_input,
    },
}, objective)
assert prepared["goal_spec"]["prompt"] == objective
assert prepared["normalization_events"][0]["to"] == "record[]"
assert "configured sourc" in prepared["text"]

# A one-goal input name is explicitly canonicalized out of deps, with raw output retained.
dependency_raw = '{"prompt_inputs":["query"],"goals":[{"id":"retrieve_records","in":["query"],"deps":["query"]}]}'
dependency_prepared = prepare_agent_candidate({
    "text": dependency_raw,
    "goal_spec": {
        "prompt": objective,
        "prompt_inputs": ["query"],
        "goals": [{
            "id": "retrieve_records", "class": "retrieve", "desc": objective,
            "in": ["query"], "out": "record[]", "eff": ["read(source)"],
            "sigma": "data", "n": 100, "exactness": "tolerant",
            "deps": ["query"], "lang": "en", "optional": False,
        }],
    },
}, objective)
assert dependency_prepared["goal_spec"]["goals"][0]["deps"] == []
assert dependency_prepared["normalization_events"][0]["rule"] == "single_goal_input_name_not_dependency"
assert dependency_prepared["text"] == dependency_raw
studio_api.validate_studio_goal_spec(dependency_prepared["goal_spec"])
print("Auditable single-goal input dependency canonicalization OK")

# Objective Contract constraints are a separate namespace from GoalSpec goal IDs.
# Remove only exact constraint-name collisions, preserve the contract and retain provenance.
constraint_dependency_raw = '{"prompt_inputs":["query"],"goals":[{"id":"g1","deps":["source_data_required"]}]}'
constraint_dependency_prepared = prepare_agent_candidate({
    "text": constraint_dependency_raw,
    "goal_spec": {
        "prompt": objective,
        "prompt_inputs": ["query"],
        "goals": [{
            "id": "g1", "class": "retrieve", "desc": objective,
            "in": ["query"], "out": "record[]", "eff": ["read(source)"],
            "sigma": "data", "n": 100, "exactness": "tolerant",
            "deps": ["source_data_required"], "lang": "en", "optional": False,
        }],
    },
}, objective, source_frame)
assert constraint_dependency_prepared["goal_spec"]["goals"][0]["deps"] == []
assert any(
    event["rule"] == "objective_constraint_name_not_goal_dependency"
    for event in constraint_dependency_prepared["normalization_events"]
)
assert source_frame["constraints"]["inputs"]["source_data_required"] is True
studio_api.validate_studio_goal_spec(constraint_dependency_prepared["goal_spec"])
assert constraint_dependency_prepared["text"] == constraint_dependency_raw
print("Objective Contract constraint/dependency namespace separation OK")

unresolved, unresolved_events = canonicalize_declared_plural_types({
    "prompt_inputs": ["text"],
    "goals": [{
        "id": "g1", "class": "retrieve", "desc": "Retrieve widgets",
        "in": ["query"], "out": "widgets[]", "deps": [],
    }],
})
assert unresolved["goals"][0]["out"] == "widgets[]"
assert unresolved_events == []
assert 'markStudioAuthRejected' in page
assert 'maxlength="12000"' in page
print("Agent objective preservation, Θ-bounded normalization, and auth recovery checks OK")

# 4) Metadata-only tool discovery through TaaS Core. No MCP invocation occurs.
DISCOVERY_SMOKE_TEST = True
if DISCOVERY_SMOKE_TEST:
    token = os.environ["TAAS_STUDIO_API_TOKEN"]
    discovery_payload = json.dumps({
        "human_review_confirmed": True,
        "goal_spec": {
            "prompt": "Retrieve records matching the provided query from the configured source.",
            "prompt_inputs": ["query"],
            "goals": [{
                "id": "g1",
                "class": "retrieve",
                "in": ["query"],
                "out": "record[]",
                "eff": ["read(sample_registry)"],
                "sigma": "data",
                "n": 1,
                "exactness": "tolerant",
                "deps": [],
                "desc": "Retrieve records matching a query from the configured source.",
            }],
        },
        "tools": [{
            "server": "sample_registry",
            "name": "lookup_records",
            "description": "Search and return records matching a query.",
            "input_schema": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        }],
    }).encode("utf-8")
    req = urllib.request.Request(
        f"http://127.0.0.1:{APP_PORT}/api/v1/objectives/discover",
        data=discovery_payload,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    discovery_result = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
    assert discovery_result["status"] in {"selected", "abstained"}
    assert discovery_result["execution_performed"] is False
    print("Core discovery smoke test:", discovery_result["status"])
    print("Selected tools:", discovery_result["selected_tools"])
    print("Hypergraph projection:", discovery_result["hypergraph_projection_status"])
    diagnostics = discovery_result.get("discovery_diagnostics")
    assert diagnostics and diagnostics.get("schema") == "taas.discovery-diagnostics/v1", discovery_result
    assert diagnostics.get("available") is True, diagnostics
    assert diagnostics["index_live_count"] == 1, diagnostics
    assert diagnostics["eligibility"]["eligible_count"] == 1, diagnostics
    assert diagnostics["eligibility"]["composition_count_matches_preflight"] is True, diagnostics
    print("Discovery pipeline stage:", diagnostics["pipeline_stage"])
    print("Discovery diagnostics:", json.dumps(diagnostics, indent=2))

# 5) Optional real-model -> GoalSpec -> Core gap-analysis test
RUN_LIVE_OBJECTIVE_TEST = False
if RUN_LIVE_OBJECTIVE_TEST:
    token = os.environ["TAAS_STUDIO_API_TOKEN"]
    request = urllib.request.Request(
        f"http://127.0.0.1:{APP_PORT}/api/v1/models/runtime",
        headers={"Authorization": f"Bearer {token}"},
    )
    runtime = json.loads(urllib.request.urlopen(request).read().decode("utf-8"))
    ollama_selected = runtime.get("providers", {}).get("ollama", {}).get("model_selected")
    provider = "ollama" if ollama_selected else "huggingface"
    loaded = ollama_selected if provider == "ollama" else runtime.get("model_loaded", False)
    if not loaded:
        print("SKIPPED: load a model in Studio, then set RUN_LIVE_OBJECTIVE_TEST=True.")
    else:
        payload = json.dumps({
            "provider": provider,
            "prompt": "Retrieve customer records and calculate the average transaction amount",
            "max_new_tokens": 1200,
        }).encode("utf-8")
        req = urllib.request.Request(
            f"http://127.0.0.1:{APP_PORT}/api/v1/objectives/prepare",
            data=payload,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        result = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
        print(json.dumps({
            "goal_spec": result.get("goal_spec"),
            "normalization_events": result.get("normalization_events"),
            "objective_contract_constraints": result.get("objective_contract", {}).get("constraints"),
            "workflow": result.get("workflow"),
            "structural_validation": result.get("structural_validation"),
            "conformance": result.get("conformance"),
            "core_analysis": result.get("core_analysis"),
            "diagnostics": result.get("semantic_diagnostics"),
        }, indent=2))
        assert result.get("workflow", {}).get("objective_interpretation") == "completed"
        assert result.get("structural_validation", {}).get("valid") is True, result.get("semantic_diagnostics")
        assert result.get("conformance", {}).get("accepted_for_human_review") is True, result.get("conformance")
        assert all(
            "source_data_required" not in goal.get("deps", [])
            for goal in result.get("goal_spec", {}).get("goals", [])
        )
        normalized_goals = result.get("goal_spec", {}).get("goals", [])
        assert len(normalized_goals) == 2, normalized_goals
        assert normalized_goals[0].get("id") == "g1" and normalized_goals[0].get("deps") == [], normalized_goals
        assert normalized_goals[1].get("id") == "g2" and normalized_goals[1].get("deps") == ["g1"], normalized_goals
        assert all("sum(record.amount)" not in effect for goal in normalized_goals for effect in goal.get("eff", [])), normalized_goals
        core = result.get("core_analysis", {})
        assert core.get("status") == "tool_and_model_capability_gaps_detected", core
        assert core.get("tool_required_goal_ids") == ["g1"], core
        assert core.get("model_capability_gap_goal_ids") == ["g2"], core
        assert core.get("gap_goal_ids") == ["g1", "g2"], core
        assert core.get("goals", [{}])[0].get("rho") == 0.0, core
        assert core.get("model_profile", {}).get("calibrated_profile_matched") is False
        assert result.get("conformance", {}).get("semantic_correctness_verified") is False
        assert result.get("workflow", {}).get("tool_execution") == "disabled"
        print("LIVE OBJECTIVE TEST: Core gap_analysis executed; proposal remains subject to human review.")
else:
    print("Live inference test is optional and currently disabled. Load a model first.")



## Objective Intelligence v3 acceptance
Run the conservative Objective Contract v3 parser, exact clause-provenance graph and explicit dependency-relation invariants, source-span checks, clarification gating and Studio API regression checks against this pinned revision. The test suite does not claim semantic correctness or authorize discovery.


In [ ]:
# Hypothesis is a declared test extra; repair a partially provisioned Colab runtime before collection.
_hypothesis_check = subprocess.run(
    [sys.executable, "-c", "import hypothesis"],
    cwd=WORKDIR, text=True, capture_output=True
)
if _hypothesis_check.returncode:
    _test_deps = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-e", ".[test]"],
        cwd=WORKDIR, text=True, capture_output=True
    )
    print(_test_deps.stdout[-3000:])
    if _test_deps.returncode:
        print(_test_deps.stderr[-8000:])
        raise RuntimeError("Could not install TaaS test dependencies (including Hypothesis)")
_hypothesis_verify = subprocess.run(
    [sys.executable, "-c", "import hypothesis; print(hypothesis.__version__)"],
    cwd=WORKDIR, text=True, capture_output=True
)
if _hypothesis_verify.returncode:
    print(_hypothesis_verify.stderr[-4000:])
    raise RuntimeError("Hypothesis is still unavailable after installing the declared test extra")
print("Hypothesis ready:", _hypothesis_verify.stdout.strip())

objective_regression_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_objective_contract_v2.py", "tests/test_objective_contract_v3.py", "-k", "oncology"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(objective_regression_tests.stdout)
if objective_regression_tests.returncode:
    print(objective_regression_tests.stderr[-8000:])
    raise RuntimeError("Oncology objective outcome/temporal regression failed")
print("Oncology objective regression passed: search + translate + extract; November 2024; top-one descending publication-date selection; provenance dependencies.")

v3_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_objective_contract_v3.py", "tests/test_studio_api.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(v3_tests.stdout)
if v3_tests.returncode:
    print(v3_tests.stderr[-8000:])
    raise RuntimeError("Objective Intelligence v3 acceptance failed")
print("Objective Intelligence v3 acceptance passed; Studio API acceptance passed.")


## Agent Release Readiness acceptance

This acceptance runs the focused release-readiness unit tests, then exercises the authenticated release-readiness endpoint against the Studio process launched above. It creates a temporary draft Agent through the API, advances it through structural validation, and checks that the report is evidence-backed and remains non-executable. No MCP tool is invoked and no human approval or assurance evidence is fabricated.

Expected result for this minimally configured candidate: `NEEDS_REVIEW`, with authorization and execution both false.

In [ ]:
import json, subprocess, time, urllib.request, urllib.error

release_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_release_readiness.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(release_tests.stdout)
if release_tests.returncode:
    print(release_tests.stderr[-10000:])
    raise RuntimeError("Agent Release Readiness unit tests failed")

API_BASE = f"http://{APP_HEALTH_HOST}:{APP_PORT}"
def studio_api_call(method, path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(
        API_BASE + path,
        data=data,
        method=method,
        headers={
            "Authorization": f"Bearer {api_token}",
            "Content-Type": "application/json",
        },
    )
    with urllib.request.urlopen(request, timeout=15) as response:
        body = response.read().decode("utf-8")
        return response.status, json.loads(body) if body else {}

agent_id = f"release-check-{int(time.time())}"
manifest = {
    "apiVersion": "taas.ai/v1alpha1",
    "kind": "Agent",
    "metadata": {
        "id": agent_id,
        "name": "Colab Release Readiness Check",
        "version": "1.0.0",
        "description": "Ephemeral API acceptance candidate",
    },
    "spec": {
        "model": {"provider": "huggingface", "model_id": "org/model"},
        "instructions": "This is a non-executing release readiness acceptance candidate.",
        "tool_allowlist": [],
        "resources": {"max_new_tokens": 64, "timeout_seconds": 30, "max_tool_calls": 0},
        "approval": {
            "require_human_approval": True,
            "require_assurance": True,
            "allow_side_effects": False,
        },
    },
}
status, created = studio_api_call("POST", "/api/v1/agents", manifest)
assert status == 201, (status, created)
status, validated = studio_api_call(
    "POST", f"/api/v1/agents/{agent_id}/versions/1.0.0/validate"
)
assert status == 200, (status, validated)
status, readiness = studio_api_call(
    "GET", f"/api/v1/agents/{agent_id}/versions/1.0.0/release-readiness"
)
assert status == 200, (status, readiness)
assert readiness["decision"] == "NEEDS_REVIEW", readiness
assert readiness["checks"]["manifest_structurally_valid"] is True, readiness
assert readiness["checks"]["manifest_integrity_verified"] is True, readiness
assert readiness["checks"]["trusted_assurance_evidence_verified"] is False, readiness
assert readiness["checks"]["human_approval_record_verified"] is False, readiness
assert readiness["authorization_granted"] is False, readiness
assert readiness["execution_enabled"] is False, readiness
print("Release readiness API: HTTP", status)
print("Decision:", readiness["decision"])
print("Blockers:", [item["code"] for item in readiness["blockers"]])
print("Authorization granted:", readiness["authorization_granted"])
print("Execution enabled:", readiness["execution_enabled"])
print("Agent Release Readiness Colab acceptance passed.")


## Agent Intelligence Context acceptance

Run focused unit tests for manifest validation, trigger-based skill selection, prompt injection, immutable Agent binding, transitive Skill dependency DAG resolution and refinement-context reuse. The optional live API check creates a temporary Agent version and sends one real objective through the loaded model. It verifies that the selected Agent instructions and matching skill were resolved and traced; it does not require the model to produce a particular semantic answer, authorize tools, or execute anything.

Set `RUN_LIVE_AGENT_CONTEXT_TEST = True` after loading a model in Studio to enable the live inference check.


In [ ]:
import json, subprocess, time

skill_registry_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_skill_registry.py", "tests/test_skill_registry_api.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(skill_registry_tests.stdout)
if skill_registry_tests.returncode:
    print(skill_registry_tests.stderr[-10000:])
    raise RuntimeError("Reusable Skill Registry acceptance tests failed")
print("Reusable Skill Registry unit and authenticated API tests passed.")

agent_skill_binding_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_agent_reusable_skill_binding.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(agent_skill_binding_tests.stdout)
if agent_skill_binding_tests.returncode:
    print(agent_skill_binding_tests.stderr[-10000:])
    raise RuntimeError("Agent reusable Skill binding acceptance tests failed")
print("Agent reusable Skill binding unit, API and Studio picker tests passed.")

skill_dependency_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_skill_dependency_graph.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(skill_dependency_tests.stdout)
if skill_dependency_tests.returncode:
    print(skill_dependency_tests.stderr[-10000:])
    raise RuntimeError("Skill dependency DAG acceptance tests failed")
print("Skill dependency DAG validation, cycle, bounds and Agent materialization tests passed.")

agent_context_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_agent_intelligence_context.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(agent_context_tests.stdout)
if agent_context_tests.returncode:
    print(agent_context_tests.stderr[-10000:])
    raise RuntimeError("Agent Intelligence Context unit tests failed")
print("Agent Intelligence Context unit tests passed.")

from taas.studio.agent_spec import validate_agent_manifest
from taas.studio.agent_context import build_agent_intelligence_context
from taas.studio.analyst_contract import build_analyst_system_prompt

context_manifest = validate_agent_manifest({
    "apiVersion": "taas.ai/v1alpha1",
    "kind": "Agent",
    "metadata": {
        "id": "colab-context-smoke",
        "name": "Colab Agent Context Smoke",
        "version": "1.0.0",
        "description": "In-process Agent Intelligence Context check",
    },
    "spec": {
        "model": {"provider": "huggingface", "model_id": "org/model"},
        "instructions": "Preserve each independently sourced fact and every explicitly requested computation.",
        "skills": [{
            "id": "date-difference",
            "name": "Date difference",
            "version": "1.0.0",
            "description": "Decompose a multi-source date comparison.",
            "instructions": "Create independent acquisition goals for both dates, then a distinct exact computation goal depending on both.",
            "trigger_phrases": ["year difference", "date difference"],
        }],
        "tool_allowlist": [],
        "resources": {"max_new_tokens": 512, "timeout_seconds": 90, "max_tool_calls": 0},
        "approval": {
            "require_human_approval": True,
            "require_assurance": True,
            "allow_side_effects": False,
        },
    },
})
resolved_context, context_trace = build_agent_intelligence_context(
    context_manifest,
    "Find two independently sourced dates and calculate their year difference.",
    manifest_sha256="c" * 64,
    lifecycle_state="draft",
)
prompt = build_analyst_system_prompt(resolved_context)
assert context_trace["instructions_injected"] is True
assert context_trace["selected_skill_ids"] == ["date-difference"]
assert "Create independent acquisition goals for both dates" in prompt
assert "FINAL NON-OVERRIDABLE CONTRACT" in prompt
print("In-process Agent instruction + skill prompt injection OK")
print("Selected skills:", context_trace["selected_skill_ids"])

RUN_LIVE_AGENT_CONTEXT_TEST = False
if RUN_LIVE_AGENT_CONTEXT_TEST:
    runtime_state_status, runtime_state = studio_api_call("GET", "/api/v1/models/runtime")
    assert runtime_state_status == 200, runtime_state
    ollama_selected = runtime_state.get("providers", {}).get("ollama", {}).get("model_selected")
    provider = "ollama" if ollama_selected else "huggingface"
    loaded = bool(ollama_selected) if provider == "ollama" else runtime_state.get("model_loaded", False)
    if not loaded:
        print("SKIPPED: load a model in Studio, then rerun with RUN_LIVE_AGENT_CONTEXT_TEST=True.")
    else:
        agent_id = f"agent-context-{int(time.time())}"
        live_manifest = {
            "apiVersion": "taas.ai/v1alpha1",
            "kind": "Agent",
            "metadata": {
                "id": agent_id,
                "name": "Colab Agent Intelligence Context",
                "version": "1.0.0",
                "description": "Temporary context-injection acceptance candidate",
            },
            "spec": {
                "model": {"provider": "huggingface", "model_id": "org/model"},
                "instructions": "Preserve all requested outcomes. Independently acquire each required fact before computing a result.",
                "skills": [{
                    "id": "year-difference",
                    "name": "Year difference",
                    "version": "1.0.0",
                    "description": "Independent date acquisition and exact computation.",
                    "instructions": "Create one independent acquisition goal for each independently sourced date, then a separate compute_exact goal depending on both.",
                    "trigger_phrases": ["year difference", "date difference"],
                }],
                "tool_allowlist": [],
                "resources": {"max_new_tokens": 1200, "timeout_seconds": 180, "max_tool_calls": 0},
                "approval": {
                    "require_human_approval": True,
                    "require_assurance": True,
                    "allow_side_effects": False,
                },
            },
        }
        create_status, created_agent = studio_api_call("POST", "/api/v1/agents", live_manifest)
        assert create_status == 201, created_agent
        prepare_status, prepared = studio_api_call(
            "POST",
            "/api/v1/objectives/prepare",
            {
                "provider": provider,
                "prompt": "Find the repository creation date and the official domain registration date, then calculate their year difference.",
                "agent_id": agent_id,
                "agent_version": "1.0.0",
                "max_new_tokens": 1200,
            },
        )
        assert prepare_status == 200, prepared
        trace = prepared.get("agent_intelligence_context", {})
        assert trace.get("status") == "injected", trace
        assert trace.get("instructions_injected") is True, trace
        assert trace.get("selected_skill_ids") == ["year-difference"], trace
        assert trace.get("manifest_sha256") == created_agent.get("manifest_sha256"), trace
        assert prepared.get("workflow", {}).get("tool_execution") == "disabled", prepared.get("workflow")
        print("Live Agent Intelligence Context API acceptance passed.")
        print("Agent:", trace.get("agent_id"), "@", trace.get("agent_version"))
        print("Selected skills:", trace.get("selected_skill_ids"))
        print("GoalSpec proposal:", json.dumps(prepared.get("goal_spec"), indent=2))
        print("Human review remains mandatory; no tool call or execution was performed.")
else:
    print("Live model call is optional and currently disabled. Enable it after loading a model.")

skill_hypergraph_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_skill_hypergraph_projection.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(skill_hypergraph_tests.stdout)
if skill_hypergraph_tests.returncode:
    print(skill_hypergraph_tests.stderr[-10000:])
    raise RuntimeError("Skill Hypergraph projection acceptance tests failed")
print("Skill DAG to Core Hypergraph projection and authenticated Studio API tests passed.")


## Tool Intelligence acceptance

Run focused unit and authenticated API tests for the Tool Intelligence slices against this pinned revision: registry resolution with definition-hash verification, explainable hybrid retrieval (component scores, RRF contributions, eligibility flags), per-tool eligibility rejection reasons under positive and closed-world policies, optional hypergraph context, and registry-backed discovery via `registry_tool_ids`. These tests inspect capability, relevance, evidence, and eligibility — none invoke tools or imply execution authorization.

In [ ]:
import json, subprocess, urllib.request, urllib.error

tool_intelligence_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_tool_registry_intelligence.py",
     "tests/test_tool_retrieval_explanation.py",
     "tests/test_tool_eligibility.py",
     "tests/test_registry_tool_discovery.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(tool_intelligence_tests.stdout)
if tool_intelligence_tests.returncode:
    print(tool_intelligence_tests.stderr[-10000:])
    raise RuntimeError("Tool Intelligence unit and API tests failed")

API_BASE = f"http://{APP_HEALTH_HOST}:{APP_PORT}"
def ti_call(method, path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        API_BASE + path, data=data, method=method,
        headers={"Authorization": f"Bearer {api_token}",
                 "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as err:
        return err.code, json.loads(err.read().decode("utf-8"))

# 1 · Registry search + capability profile with hash verification.
status, tools = ti_call("GET", "/api/v1/tools?q=read&limit=5")
assert status == 200, tools
assert tools["tools"], tools
first = tools["tools"][0]
server, name = first["tool_id"].split("::", 1)
status, profile = ti_call("GET", f"/api/v1/tools/{server}/{name}/intelligence")
assert status == 200, profile
assert profile["identity"]["definition_hash_verified"] is True, profile
assert profile["evidence_summary"]["declared"]["annotations_are_trusted_evidence"] is False

# 2 · Explainable retrieval: component scores + eligibility flags.
status, retrieval = ti_call("POST", "/api/v1/tools/retrieve", {
    "query": "read records", "k": 5,
    "policy": {"required": ["read_only"], "lmin": 0.3}})
assert status == 200, retrieval
assert retrieval["schema"] == "taas.retrieval-explanation/v1"
assert retrieval["execution_performed"] is False
top = retrieval["results"][0]
assert top["rrf_score"] > 0
assert set(top["components"]) <= {"lex", "den", "str"}

# 3 · Eligibility explanation for a closed-world policy.
status, elig = ti_call("POST", "/api/v1/tools/eligibility", {
    "policy": {"forbidden": ["writes"], "encoding": "closed_world", "lmin": 0.3}})
assert status == 200, elig
assert elig["schema"] == "taas.eligibility-explanation/v1"
assert elig["summary"]["evaluated"] == len(elig["tools"])
assert elig["execution_performed"] is False

# 4 · Hypergraph context folds into the retrieve response on request.
status, hg = ti_call("POST", "/api/v1/tools/retrieve", {
    "query": "read records", "k": 3, "include_hypergraph": True})
assert status == 200, hg
projection = hg["hypergraph_projection"]
assert projection["schema"] == "taas.hypergraph-projection/v1"
assert projection["vertices"]

# 5 · Registry-backed discovery: exact ids resolve; unresolved ids are rejected.
status, missing = ti_call("POST", "/api/v1/objectives/discover", {
    "goal_spec": {"prompt": "Read records", "prompt_inputs": ["text"],
                  "goals": [{"id": "g1", "class": "retrieve", "in": ["query"],
                             "out": "record[]", "eff": ["read(records)"],
                             "sigma": "data", "deps": [], "desc": "Read records"}]},
    "human_review_confirmed": True,
    "registry_tool_ids": ["ghost::missing"]})
assert status == 422, missing
print("Tool Intelligence acceptance OK — no tool was invoked and nothing executed.")

## Governance & persistence acceptance

Run focused suites for the governance and persistence slices against this pinned revision: tool comparison (structural diffs, score parity), observation-evidence wiring (JSONL obs/att store upgrades contract provenance), durable discovery-run records (persist → list → fetch, bounded retention), the hash-chained audit ledger (link integrity + tamper detection + verify endpoint), and approval ceremony hardening (same-assessor replay → 409, distinct assessor allowed, expiry enforced, ceremony event in the ledger). These tests verify recorded governance state — none imply execution occurred.

In [ ]:
import json, subprocess, urllib.request, urllib.error

governance_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_tool_compare.py",
     "tests/test_obs_evidence_wiring.py",
     "tests/test_discovery_runs.py",
     "tests/test_audit_ledger.py",
     "tests/test_agent_release_approvals.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(governance_tests.stdout)
if governance_tests.returncode:
    print(governance_tests.stderr[-10000:])
    raise RuntimeError("Governance & persistence unit/API tests failed")

API_BASE = f"http://{APP_HEALTH_HOST}:{APP_PORT}"
def gov_call(method, path, payload=None):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(
        API_BASE + path, data=data, method=method,
        headers={"Authorization": f"Bearer {api_token}",
                 "Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.status, json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as err:
        return err.code, json.loads(err.read().decode("utf-8"))

# 1 · Tool comparison: structural diff over exact registry ids.
status, tools = gov_call("GET", "/api/v1/tools?limit=3")
assert status == 200 and len(tools["tools"]) >= 2, tools
ids = [r["tool_id"] for r in tools["tools"][:2]]
status, cmp_result = gov_call("POST", "/api/v1/tools/compare", {
    "tool_ids": ids, "query": "read records"})
assert status == 200, cmp_result
assert cmp_result["schema"] == "taas.tool-compare/v1"
assert cmp_result["execution_performed"] is False
assert cmp_result["summary"]["compared"] == 2

# 2 · Discovery-run persistence: run records are observable, not executions.
status, discover = gov_call("POST", "/api/v1/objectives/discover", {
    "goal_spec": {"prompt": "Read records", "prompt_inputs": ["text"],
                  "goals": [{"id": "g1", "class": "retrieve", "in": ["query"],
                             "out": "record[]", "eff": ["read(records)"],
                             "sigma": "data", "deps": [], "desc": "Read records"}]},
    "human_review_confirmed": True,
    "registry_tool_ids": ids[:1]})
assert status == 200, discover
run_id = discover["discovery_run_id"]
assert run_id and discover["discovery_run_persisted"] is True
status, run = gov_call("GET", f"/api/v1/discovery/runs/{run_id}")
assert status == 200 and run["execution_record"] is False
status, listing = gov_call("GET", "/api/v1/discovery/runs?limit=5")
assert any(r["run_id"] == run_id for r in listing["runs"])

# 3 · Audit ledger: chain is valid and the run event was recorded.
status, verify = gov_call("GET", "/api/v1/audit/verify")
assert status == 200, verify
assert verify["chain_valid"] is True
status, events = gov_call("GET", "/api/v1/audit/events?event_type=discovery_run")
assert events["count"] >= 1
print("Governance & persistence acceptance OK — records are observations, never execution proof.")

## F01 — Platform Overview acceptance

This acceptance suite validates the API-backed Platform Overview added to the extension branch. It runs against the exact branch revision checked out in the setup cell.


In [ ]:
platform_overview_tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_platform_overview.py"],
    cwd=WORKDIR, text=True, capture_output=True
)
print(platform_overview_tests.stdout)
if platform_overview_tests.returncode:
    print(platform_overview_tests.stderr)
    raise RuntimeError("F01 Platform Overview acceptance failed")
print("F01 Platform Overview acceptance passed.")


## Stop Studio

Run this cell when finished. Colab storage and runtime memory are temporary.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try:
        studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill()
        studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.flush()
    studio_log.close()
if "tunnel_process" in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate()
    try:
        tunnel_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        tunnel_process.kill()
        tunnel_process.wait(timeout=5)
if "tunnel_log" in globals() and not tunnel_log.closed:
    tunnel_log.flush()
    tunnel_log.close()
if "ollama_process" in globals() and ollama_process.poll() is None:
    ollama_process.terminate()
    try:
        ollama_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        ollama_process.kill()
        ollama_process.wait(timeout=5)
print("TaaS Studio, optional tunnel and optional Ollama daemon stopped.")
